# Qt–Ft Agglomeration Simulation — Run Notebook

Coarse-grained ReaDDy2 Brownian-dynamics simulation of Qt encapsulin and ferritin (Ft) agglomeration.

**This notebook only runs simulations.** All parameters live in the single **Configuration** cell; the single **Run** cell then executes any combination of:

- `RUN_MODE = "single"` (one trajectory) or `"ensemble"` (multi-replica)
- `ENABLE_DEAGG = False` (plain agglomeration) or `True` (agglomeration ↔ deagglomeration cycling)

Plotting and analysis live in the separate `Plot_Ensemble_Results_*.ipynb` notebooks.

## 1. Imports and Setup

In [1]:
import os
import sys
import readdy

# qtft package. Plotting lives in the Plot_Ensemble_Results_* notebooks, so it is
# deliberately not imported here; `analysis` is only used for the optional summary/XYZ export.
import qtft as sim
import qtft.analysis as analysis
from qtft import EnsembleSimulation

print(f"ReaDDy: {readdy.__version__}")
print(f"Python: {sys.version}")

ReaDDy: 2.0.13-5
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]


## 2. Module Reload Helper

In [2]:
import importlib
import qtft, qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble

# Reload submodules in dependency order (restart the kernel for deep changes).
for _m in (qtft.config, qtft.system, qtft.engine, qtft.analysis, qtft.ensemble, qtft):
    importlib.reload(_m)

sim = qtft
analysis = qtft.analysis
from qtft import EnsembleSimulation
print("Modules reloaded")

Modules reloaded


## 3. Configuration — all parameters

In [ ]:
# ============================================================
# SIMULATION CONFIGURATION  —  all parameters live here
# ============================================================
# Edit parameters, run this cell, then run the "Run" cell below.
#
# >>> SOFT-MODE LONG-TIMESTEP PRESET (1 ms timestep, 5 min simulated time) <<<
# Uses the "soft" harmonic-repulsion potential (README section 12) to take a huge
# integration step. This is a HEAVILY COARSE-GRAINED regime — read before running:
#   * dt = 1e6 ns (1 ms);  n_steps = 300_000  ->  300_000 x 1 ms = 300 s = 5.00 min.
#   * Diffusion is lowered to ~1e-5 nm^2/ns so the per-step displacement sqrt(2*D*dt)
#     (~4.5 nm) stays well below the particle/binding-radius scales. Real Qt/Ft diffuse
#     ~5 orders of magnitude faster, so "5 min" is MODEL time under this slow-particle
#     assumption, not literal wall-clock or real-particle time.
#   * k_bond = 0.01 (soft) is REQUIRED for 1 ms stability (k_bond*D must stay small);
#     clusters are correspondingly floppy (bond length ~ r0 +/- ~10-15 nm).
#   * kon = 1e-7 keeps the per-step binding probability p = 1 - exp(-kon*dt) ~ 0.1
#     (stochastic regime); a faster rate cannot be resolved at a 1 ms step.
#   * equilibration_potential = "soft": MANDATORY here. WCA/LJ equilibration would blow
#     up at a 1 ms step. (Soft repulsion also tolerates initial overlaps.)
# Calibrate before long runs with:  python scripts/calibrate_timestep.py
# TIP: pilot with n_steps = 5_000 first (confirm stability + that clusters nucleate),
#      then scale up to 300_000.

# ----- What to run -----
RUN_MODE     = "single"    # "single" (one trajectory) or "ensemble" (multi-replica)
ENABLE_DEAGG = False       # False = plain agglomeration; True = agglomeration <-> deagglomeration cycling

# ----- Output locations -----
SINGLE_RUN_ROOT = "Simulation_Files_Single_Runs"  # parent folder for single runs
ENSEMBLE_ROOT   = "Simulation_Files_Ensembles"    # parent folder for ensembles

# ----- Output toggles -----
SAVE_CONFIG = True    # write <name>_config.json next to the trajectory (single runs; ensembles always save it)
EXPORT_XYZ  = True    # also export an OVITO-friendly .xyz after the run

# ----- Ensemble settings (used when RUN_MODE == "ensemble") -----
N_REPLICAS          = 10
PARALLEL            = True
N_WORKERS           = 10
EQUILIBRATION_STEPS = 1000    # runs under equilibration_potential="soft"; soft mode needs little relaxation

# ----- Deagglomeration / cycling settings (used when ENABLE_DEAGG) -----
KOFF            = 1e-7        # per-edge bond-breaking rate (1/ns); tuned so p_break stays stochastic at 1 ms
AGG_STEPS       = 150_000     # steps of each agglomeration phase   (x 1 ms = 2.5 min)
DEAGG_STEPS     = 150_000     # steps of each deagglomeration phase (x 1 ms = 2.5 min)
N_CYCLES        = 1           # number of agg->deagg cycles (total phases = 2 * N_CYCLES)
AGG_POTENTIAL   = "soft"      # soft mode throughout (WCA/LJ would be unstable at a 1 ms step)
DEAGG_POTENTIAL = "soft"

# ============================================================
# Build the configuration
# ============================================================
config = sim.SimulationConfig(
    # ----- Particle properties (diffusion lowered for 1 ms stability) -----
    qt=sim.ParticleConfig(
        name="Qt",
        radius=21.0,             # nm (encapsulin)
        diffusion=5e-6,          # nm^2/ns
        cluster_diffusion=3e-6,  # nm^2/ns (when bound in a cluster)
    ),
    ft=sim.ParticleConfig(
        name="Ft",
        radius=6.0,              # nm (ferritin)
        diffusion=1e-5,          # nm^2/ns
        cluster_diffusion=7e-6,  # nm^2/ns (when bound in a cluster)
    ),

    # ----- Topology / binding -----
    topology=sim.TopologyConfig(
        name="QtFt_Cluster",
        binding_radius=27.0,    # nm (~ r_Qt + r_Ft + buffer)
        kon=1e-9,                # microscopic binding rate (1/ns); MAX faithful rate at a 1 ms step
        k_bond=0.1,             # kJ/(mol*nm^2) SOFT bond — required for 1 ms stability
        ft_monovalent=False,     # True -> Ft caps at 1 bond (single-Qt-star clusters); adds _FtMono tag
    ),

    # ----- Potential selector -----
    # In soft mode the LJ epsilon values are IGNORED (only lj.potential_type is read);
    # excluded volume comes entirely from the per-pair soft.k_* below. The epsilons are
    # kept here so you can toggle potential_type to "LJ"/"WCA" without re-entering them.
    lj=sim.LennardJonesConfig(
        epsilon_QtQt=1.5,        # ignored in soft mode (used only for "LJ"/"WCA")
        epsilon_FtFt=1.5,        # ignored in soft mode
        epsilon_QtFt=3.0,        # ignored in soft mode
        potential_type="soft",   # "soft" (harmonic repulsion) | "LJ" (attractive) | "WCA" (repulsive)
    ),

    # ----- Soft excluded volume: PER-PAIR harmonic repulsion (no attractive well) -----
    # Force constants in kJ/(mol*nm^2); 0 disables a pair. The thermal overlap scale is
    # ~ sqrt(2*kB*T / k), so the small Ft needs a STIFFER constant than Qt to stop
    # interpenetrating. The values below reproduce the previous uniform 0.005; to reduce
    # Ft overlap, raise k_FtFt / k_QtFt (e.g. k_FtFt=1.0, k_QtFt=0.5) — but stiffer k at a
    # 1 ms step needs lower D, so re-check stability with scripts/calibrate_timestep.py.
    # Cluster/mixed pairs cascade from these three free-free values unless set explicitly.
    soft=sim.SoftPotentialConfig(
        k_QtQt=0.5,   # Qt-Qt
        k_FtFt=0.5,   # Ft-Ft  (raise to stiffen small-particle repulsion / reduce overlap)
        k_QtFt=0.5,   # Qt-Ft  (raise to stiffen)
    ),

    # ----- Equilibration (reactions always off) -----
    equilibration_potential="soft",   # MUST be "soft" at a 1 ms step (WCA/LJ would blow up)

    # ----- Simulation box -----
    box_size=(500, 500, 500),    # nm
    periodic_boundary=True,
    temperature=300.0,           # K

    # ----- Integration -----
    timestep=1e5,                # ns  = 1 ms
    n_steps=300_000,             # x 1 ms = 300 s = 5 min  (ignored when ENABLE_DEAGG)

    # ----- Recording -----
    record_stride=100,                  # save trajectory every N steps (-> 3000 frames)
    observable_stride=100,              # record observables every N steps
    particles_observable_stride=1000,   # per-particle positions cadence (None to disable, saves disk)

    # ----- Particle counts -----
    n_qt=200,
    n_ft=400,

    # ----- Execution -----
    kernel="CPU",
    n_threads=4,
    rng_seed=22,

    # output_file is auto-generated from the parameters (see qtft.format_param_string).
)

# ----- Apply deagglomeration cycling (centralizes the phase knobs set above) -----
if ENABLE_DEAGG:
    config.topology.koff = KOFF
    config.phases = sim.make_agg_deagg_phases(
        agg_steps=AGG_STEPS,
        deagg_steps=DEAGG_STEPS,
        n_cycles=N_CYCLES,
        agg_potential=AGG_POTENTIAL,
        deagg_potential=DEAGG_POTENTIAL,
    )

config.print_summary()

SIMULATION CONFIGURATION

Particles:
  Qt: r=21.0 nm, D=5e-06 nm²/ns (cluster: D=3e-06)
  Ft: r=6.0 nm, D=1e-05 nm²/ns (cluster: D=7e-06)
  Counts: 200 Qt + 400 Ft = 600 total

Topology:
  Binding radius: 27.0 nm
  Binding rate (kon): 1e-07 nm³/(ns·part)
  Bond-breaking rate (koff): 0.0 /(edge·ns)
  Bond stiffness: 0.1 kJ/(mol·nm²)
  Equilibrium bond length: 27.0 nm

Lennard-Jones:
  Potential type: soft
  Cutoff factor: 1.122
  ε Qt-Qt: 1.5 kJ/mol
  ε Ft-Ft: 1.5 kJ/mol
  ε Qt-Ft: 3.0 kJ/mol
  Cluster/mixed ε: same as free (default)

Simulation:
  Box: 500 × 500 × 500 nm
  Temperature: 300.0 K
  Equilibration potential: soft
  Timestep: 100000.0 ns (100000000.00 ps)
  Steps: 300,000 (30000000.0 µs total)
  Output: 200Qt_400Ft_soft_kQQ0.05_kFF0.05_kQF0.05_kon1e-07_dt100000000ps_30000000us.h5


## 4. Run

In [4]:
# ============================================================
# RUN  —  dispatches on RUN_MODE and ENABLE_DEAGG (set in the config cell)
# ============================================================

# Auto-named basename from the parameters (idempotent: safe to re-run this cell).
base_name = sim.format_param_string(config) + ".h5"

if RUN_MODE == "single":
    # Collect all output for this run in its own subfolder under SINGLE_RUN_ROOT.
    RUN_DIR = os.path.join(SINGLE_RUN_ROOT, base_name[:-3])
    config.output_file = os.path.join(RUN_DIR, base_name)

    result = sim.run_one(config, equilibration_steps=EQUILIBRATION_STEPS)

    if config.phases:
        # Phased run: result is a list of per-phase dicts; the stitched whole-cycle
        # trajectory path is under result[0]["combined"].
        combined = result[0].get("combined")
        export_src = combined
        print(f"Phased run complete ({len(result)} phases). Combined trajectory: {combined}")
    else:
        # Plain run: result is a readdy.Trajectory.
        export_src = config.output_file
        analysis.print_analysis_summary(config.output_file, config)

    if SAVE_CONFIG:
        cfg_path = config.output_file[:-3] + "_config.json"
        config.save_json(cfg_path)
        print(f"Saved config: {cfg_path}")

    if EXPORT_XYZ and export_src and os.path.exists(export_src):
        xyz_path = export_src.replace(".h5", ".xyz")
        analysis.convert_h5_to_xyz(export_src, xyz_path, config, overwrite=True)
        print(f"Exported XYZ: {xyz_path}")

elif RUN_MODE == "ensemble":
    ensemble = EnsembleSimulation(
        base_config=config,
        n_replicas=N_REPLICAS,
        base_dir=ENSEMBLE_ROOT,
    )
    print(f"Seeds: {ensemble.seeds}")
    ensemble.run_local(
        parallel=PARALLEL,
        n_workers=N_WORKERS,
        overwrite=True,
        equilibration_steps=EQUILIBRATION_STEPS,
    )
    ensemble.print_summary()

    if EXPORT_XYZ:
        rep_cfg = ensemble.replica_configs[0]
        if config.phases:
            src = os.path.join(rep_cfg.phase_base_dir, "trajectory_combined.h5")
        else:
            src = rep_cfg.output_file
        if os.path.exists(src):
            xyz_path = src.replace(".h5", ".xyz")
            analysis.convert_h5_to_xyz(src, xyz_path, rep_cfg, overwrite=True)
            print(f"Exported XYZ (replica_000): {xyz_path}")

else:
    raise ValueError(f"RUN_MODE must be 'single' or 'ensemble', got {RUN_MODE!r}")


EQUILIBRATION
  Running 1,000 steps without reactions (soft potential)

✓ Species: Qt, Ft, QtC, FtC
✓ soft (harmonic-repulsion) potentials (10 registered): k_QQ=0.05, k_FF=0.05, k_QF=0.05
✓ Topology 'QtFt_Cluster': k_bond=0.1; bonds only, no reactions
✓ System created: 500×500×500 nm box (equilibration mode - no reactions)
✓ Placed 200 Qt + 400 Ft particles (random)
  Running equilibration...
Configured kernel context with:
--------------------------------
 - kBT = 2.4361377615198827
 - periodic b.c. = (true, true, true)
 - box size = (500, 500, 500)
 - particle types:
     * Topology particle type "FtC" with D=7e-06
     * Topology particle type "QtC" with D=3e-06
     * Topology particle type "Ft" with D=1e-05
     * Topology particle type "Qt" with D=5e-06
 - potentials of order 2:
     * for types "QtC" and "Ft"
         * Harmonic repulsion with Force constant k=0.05
     * for types "Qt" and "QtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "Qt" and

100%|██████████| 100/100 [00:00<00:00, 143.07it/s]

[2026-07-18 18:00:40] [info] Simulation completed

EQUILIBRATION COMPLETE
  Retrieved 200 Qt + 400 Ft positions

✓ Species: Qt, Ft, QtC, FtC
✓ soft (harmonic-repulsion) potentials (10 registered): k_QQ=0.05, k_FF=0.05, k_QF=0.05
✓ Topology 'QtFt_Cluster': k_bond=0.1; 4 binding spatial reactions (kon=1e-07, binding_radius=27.0 nm, ft_monovalent=False)
✓ System created: 500×500×500 nm box
✓ Observables registered (stride=100, forces/virial stride=10000, particles observable stride=1000)
✓ Simulation created: CPU kernel, 4 threads
✓ Placed 200 Qt (provided) + 400 Ft (provided) particles

RUNNING SIMULATION
  Particles: 200 Qt + 400 Ft


  Duration: 30000000.0 µs (300,000 steps)

Configured kernel context with:
--------------------------------
 - kBT = 2.4361377615198827
 - periodic b.c. = (true, true, true)
 - box size = (500, 500, 500)
 - particle types:
     * Topology particle type "FtC" with D=7e-06
     * Topology particle type "QtC" with D=3e-06
     * Topology particle type "Ft" with D=1e-05
     * Topology particle type "Qt" with D=5e-06
 - potentials of order 2:
     * for types "QtC" and "Ft"
         * Harmonic repulsion with Force constant k=0.05
     * for types "Qt" and "QtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "Qt" and "FtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "Ft" and "FtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "QtC" and "FtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "FtC" and "FtC"
         * Harmonic repulsion with Force constant k=0.05
     * for types "QtC

100%|██████████| 30000/30000 [05:31<00:00, 90.40it/s]

[2026-07-18 18:06:12] [info] Simulation completed

SIMULATION COMPLETE



  Bond counting: Method 1 (topology.edges) - exact count

SIMULATION RESULTS SUMMARY

Initial state (t=0):
  Topologies: 600
  Average size: 1.00 particles
  Largest: 1 particles
  Bonds: 0

Final state (t=30000000.00 µs):
  Topologies: 1
  Average size: 600.00 particles
  Largest: 600 particles
  Bonds: 599

Final size distribution:
  Median: 600.0
  Mean: 600.0
  Std: 0.0
  Range: 600 - 600

Particle distribution:
  Monomers (1): 0 particles (0.0%)
  Small (2-12): 0 particles (0.0%)
  Medium (13-60): 0 particles (0.0%)
  Large (61-150): 0 particles (0.0%)
  Very large (>150): 600 particles (100.0%)

✓ Configuration saved to Simulation_Files_Single_Runs/200Qt_400Ft_soft_kQQ0.05_kFF0.05_kQF0.05_kon1e-07_dt100000000ps_30000000us/200Qt_400Ft_soft_kQQ0.05_kFF0.05_kQF0.05_kon1e-07_dt100000000ps_30000000us_config.json
Saved config: Simulation_Files_Single_Runs/200Qt_400Ft_soft_kQQ0.05_kFF0.05_kQF0.05_kon1e-07_dt100000000ps_30000000us/200Qt_400Ft_soft_kQQ0.05_kFF0.05_kQF0.05_kon1e-07_dt10000

## 5. Cluster (SLURM) execution — optional

For HPC runs, generate SLURM job-array scripts instead of running locally. This builds an `EnsembleSimulation` from the **Configuration** cell above (set the params there first), then writes `submit_ensemble.slurm` and `submit_analysis.slurm` into the ensemble directory. Nothing runs locally — submit them with `sbatch`.

In [ ]:
# ============================================================
# OPTIONAL: generate SLURM scripts for cluster execution
# ============================================================
# Builds the ensemble from the config above; does not run anything locally.
ensemble = EnsembleSimulation(
    base_config=config,
    n_replicas=N_REPLICAS,
    base_dir=ENSEMBLE_ROOT,
)

# ----- SLURM script for running replica simulations -----
ensemble.generate_slurm_scripts(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="08:00:00",             # (optional, str) Wall time limit per replica (HH:MM:SS)
    cpus_per_task=12,            # (optional, int) CPUs per replica
    memory="32G",                # (optional, str) Memory per replica

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)

# ----- SLURM script for post-simulation analysis -----
ensemble.generate_analysis_slurm_script(
    # --- SLURM job settings ---
    partition="cm4_tiny",        # (required, str) SLURM partition name
    cluster="cm4",               # (optional, str) SLURM cluster name, None to omit
    qos="cm4_tiny",              # (optional, str) Quality of service, None to omit
    time="04:00:00",             # (optional, str) Wall time limit (HH:MM:SS)
    cpus_per_task=4,             # (optional, int) CPUs for parallel analysis
    memory="32G",                # (optional, str) Memory allocation

    # --- Conda environment ---
    conda_base="<YOUR_CONDA_PATH>",  # (required, str) Full path to conda installation, e.g. "/home/user/miniconda3"
    conda_env="readdy",              # (optional, str) Name of conda environment with ReaDDy

    # --- Paths and analysis settings ---
    scripts_dir="~/Readdy_Simulations",  # (optional, str) Directory where Python scripts are located
    stride=10,                           # (optional, int) Analyze every Nth frame for structural analysis

    # --- Email notifications ---
    mail_user=None,              # (optional, str) Email for notifications, e.g. "user@example.com"
    mail_type="ALL",             # (optional, str) When to send emails: NONE, BEGIN, END, FAIL, ALL
)